## Veamos que paneles fotovoltaicos poner en cada estación y cuanta energía nos da cada uno por hora

In [29]:
import pandas as pd


tamaño = pd.read_json("optimos_semanales.json")
centroids= pd.read_csv("centroides2.csv")
centroids

,Unnamed: 0,cluster,lat,lon,num_points,adjustment_status
0,0,0.0,41.881373,12.510368,481.0,Moved out of Zone
1,1,1.0,41.901050,12.465224,351.0,Moved out of Zone
2,2,2.0,41.856676,12.477809,287.0,Moved out of Zone
3,3,3.0,41.913563,12.520829,242.0,Moved out of Zone
4,4,4.0,41.876901,12.465899,235.0,Moved out of Zone
...,...,...,...,...,...,...
185,185,185.0,41.892504,12.509260,213.0,Moved out of Zone
186,186,186.0,41.880550,12.454836,225.0,Moved out of Zone
187,187,187.0,41.905546,12.516645,94.0,Moved out of Zone
188,188,188.0,41.873436,12.512979,222.0,Moved out of Zone


In [30]:
import re
def extraer_primer_numero(texto):
    """Extrae el primer número entero o flotante dentro de los corchetes."""
    # Busca el patrón: '[' seguido de un número (entero o decimal)
    match = re.search(r'\[(\d+(\.\d+)?)', str(texto))
    if match:
        # Devuelve el valor capturado como entero
        return int(float(match.group(1)))
    return None # En caso de que el formato no coincida

# Crea la nueva columna 'tamaño' aplicando la función a la primera columna de datos
tamaño['tamaño'] = tamaño.iloc[:, 0].apply(extraer_primer_numero)


# --- PASO 2: Extraer el número de cluster del índice ---

# Usamos la función str.extract para obtener los dígitos del índice.
# r'(\d+)$' busca uno o más dígitos (\d+) al final ($) de la cadena y los captura.
tamaño['cluster'] = tamaño.index.str.extract(r'(\d+)$').astype(int)


# --- PASO 3: Crear el nuevo DataFrame final ---

# Seleccionar solo las columnas 'cluster' y 'tamaño' y establecer 'cluster' como índice
df_final = tamaño[['cluster', 'tamaño']].set_index('cluster')

In [31]:
tamaño = df_final.reset_index(drop=True)

In [33]:
# 1. Crear una clave de fusión en el DataFrame 'tamaño' (el índice 0-189)
# Esta clave debe coincidir con la columna 'cluster' del DF de coordenadas.
tamaño['cluster'] = tamaño.index

# 2. Realizar la fusión (Merge)
# Fusionamos 'df_coordenadas' con 'tamaño' usando la clave 'cluster'.
estaciones = pd.merge(
    centroids[['cluster', 'lat', 'lon']], # Coordenadas y clave
    tamaño[['cluster', 'tamaño']],             # Tamaño y clave
    on='cluster',                              # La columna de coincidencia
    how='inner'                                # Solo filas que coinciden en ambos
)


estaciones = estaciones[['tamaño', 'lat', 'lon']]

print("DataFrame 'centroids' final (tamaño, lat, lon):")
centroids.head()
print(f"\nDimensiones del DataFrame centroids: {centroids.shape}")

DataFrame 'centroids' final (tamaño, lat, lon):

Dimensiones del DataFrame centroids: (190, 6)


Tenemos la información de la estación (Ubicación y puntos de recarga máxima) Sacada de la simulación

In [ ]:
estaciones

Hagamos un mapeo para asignar a cada estación un panel fotovoltaico según cada tamaño. Especificaciones sacadas del pdf de la empresa.

In [48]:
import pandas as pd
import numpy as np

# --- ASUMIMOS que el DataFrame 'estaciones' ya está cargado con las columnas 'tamaño', 'lat', 'lon' ---
# Nota: La estructura de tu DataFrame 'estaciones' es crucial para el funcionamiento.
# ------------------------------------------------------------------------------------------------------

# --- 1. Definición de Especificaciones Eléctricas y Clases ---

# Datos de la imagen y de tu descripción
ESPECIFICACIONES_HELIO = {
    # El tamaño (m2) no se usa en la API, pero es útil para la documentación interna
    'Helios II': {'patinetes_min': 4, 'patinetes_max': 7, 'tamano_m2': 5.6, 'Potencia_Wp': 950, 'Capacidad_Wh': 2400},
    # Nota: Los límites de Helios II y IV se ajustaron para evitar solapamientos con la lógica gradual de Helios III
    'Helios III': {'patinetes_min': 8, 'patinetes_max': 12, 'tamano_m2': 6.9, 'Potencia_Wp': 1150, 'Capacidad_Wh': 2400},
    'Helios IV': {'patinetes_min': 13, 'patinetes_max': 16, 'tamano_m2': 6.9, 'Potencia_Wp': 1150, 'Capacidad_Wh': 2400}
}

# --- 2. Función para Asignar Tipo de Placa ---

def asignar_tipo_y_potencia(row):
    """
    Asigna el tipo de placa y la Potencia_Wp basada en el número de patinetes ('tamaño').
    Implementa la progresión gradual para Helios III (8 a 12 patinetes).
    """
    num_patinetes = row['tamaño']

    # Lógica de PROGRESIÓN GRADUAL (Helios III)
    if num_patinetes >= 8 and num_patinetes <= 12:
        tipo = 'Helios III'

        # Patinetes: 8, 9, 10, 11, 12 -> Potencia: 950, 1000, 1050, 1100, 1150
        potencia_wp = 950 + 50 * (num_patinetes - 8)

    # Lógica de Potencia Fija (Helios II: 4 a 7 patinetes)
    elif num_patinetes >= ESPECIFICACIONES_HELIO['Helios II']['patinetes_min'] and num_patinetes <= ESPECIFICACIONES_HELIO['Helios II']['patinetes_max']:
        tipo = 'Helios II'
        potencia_wp = ESPECIFICACIONES_HELIO['Helios II']['Potencia_Wp']

    # Lógica de Potencia Fija (Helios IV: 13 a 16 patinetes)
    elif num_patinetes >= ESPECIFICACIONES_HELIO['Helios IV']['patinetes_min']:
        tipo = 'Helios IV'
        potencia_wp = ESPECIFICACIONES_HELIO['Helios IV']['Potencia_Wp']

    return pd.Series([tipo, potencia_wp])

# ---------------------------------------------------------------------------------
# *** CORRECCIÓN CRÍTICA EN EL PASO 3 ***
# Asignación correcta de los dos valores devueltos por la función (tipo y potencia_wp)
# ---------------------------------------------------------------------------------

# Aplicar la función y expandir los dos valores de retorno a dos nuevas columnas
estaciones[['tipo_placa', 'Potencia_Wp']] = estaciones.apply(asignar_tipo_y_potencia, axis=1)


# --- 4. Asignar Especificaciones Eléctricas de la API (CORREGIDO) ---

# Creamos las columnas necesarias para la API (Potencia y Capacidad)

# Potencia FOTOVOLTAICA INSTALADA (en Wp, que debe ser kW para PVWatts)
# *** CORRECCIÓN: Usa la nueva columna 'Potencia_Wp' ya calculada, no el diccionario ESPECIFICACIONES_HELIO ***
estaciones['Potencia_kW'] = estaciones['Potencia_Wp'] / 1000

# Capacidad de la Batería (Wh, no necesaria para la API de PVWatts, pero útil)
# Se sigue usando el diccionario para datos fijos como la capacidad y el m2
estaciones['Capacidad_Bateria_Wh'] = estaciones['tipo_placa'].apply(
    lambda x: ESPECIFICACIONES_HELIO.get(x, {}).get('Capacidad_Wh', 0)
)

# Columna con el tamaño de la placa (en m2, para referencia)
estaciones['Tamano_Placa_m2'] = estaciones['tipo_placa'].apply(
    lambda x: ESPECIFICACIONES_HELIO.get(x, {}).get('tamano_m2', np.nan)
)

# --- 5. Preparar la Columna de Parámetros de la API PVWatts ---

# La columna 'PVWatts_Params' contendrá un diccionario con los parámetros listos
# para ser insertados en la URL de la API.

def preparar_parametros_api(row):
    """Genera un diccionario de parámetros PVWatts para cada estación."""
    params = {
        # Parámetros variables por estación:
        'lat': row['lat'],
        'lon': row['lon'],
        # La Potencia FOTOVOLTAICA INSTALADA (Wp) se convierte a kW (system_capacity)
        'system_capacity': row['Potencia_kW'],

        # Parámetros Fijos (Ejemplo, ajusta estos valores según la realidad)
        'module_type': 1,            # 1: Módulo Premium
        'array_type': 0,             # 0: Fijo (Estructura Abierta) - Asumimos instalación en suelo
        'tilt': 0,                   # 0: Placas Planas - Asumimos placas tumbadas
        'azimuth': 180,              # Azimuth 180 (Sur) - Aunque no afecta con tilt=0, es mejor mantener 180 que 0
        'losses': 10.0,              # Pérdidas del sistema
        'timeframe': 'hourly'
    }
    return params

# Aplicar la función para crear la columna de parámetros
estaciones['PVWatts_Params'] = estaciones.apply(preparar_parametros_api, axis=1)


# --- 6. Mostrar el Resultado ---
print("DataFrame 'estaciones' con Especificaciones y Parámetros de la API (CORREGIDO):")
estaciones[['tamaño', 'tipo_placa', 'Potencia_Wp', 'Potencia_kW', 'lat', 'lon', 'PVWatts_Params']]

DataFrame 'estaciones' con Especificaciones y Parámetros de la API (CORREGIDO):


,tamaño,tipo_placa,Potencia_Wp,Potencia_kW,lat,lon,PVWatts_Params
0,16,Helios IV,1150.0,1.15,41.881373,12.510368,"{'lat': 41.881373, 'lon': 12.5103677, 'system_..."
1,16,Helios IV,1150.0,1.15,41.901050,12.465224,"{'lat': 41.9010498, 'lon': 12.4652242, 'system..."
2,12,Helios III,1150.0,1.15,41.856676,12.477809,"{'lat': 41.8566758, 'lon': 12.4778092, 'system..."
3,11,Helios III,1100.0,1.10,41.913563,12.520829,"{'lat': 41.9135628, 'lon': 12.5208295, 'system..."
4,11,Helios III,1100.0,1.10,41.876901,12.465899,"{'lat': 41.8769013, 'lon': 12.4658987, 'system..."
...,...,...,...,...,...,...,...
185,16,Helios IV,1150.0,1.15,41.892504,12.509260,"{'lat': 41.8925043, 'lon': 12.5092605, 'system..."
186,11,Helios III,1100.0,1.10,41.880550,12.454836,"{'lat': 41.8805503, 'lon': 12.454836, 'system_..."
187,6,Helios II,950.0,0.95,41.905546,12.516645,"{'lat': 41.9055458, 'lon': 12.5166454, 'system..."
188,7,Helios II,950.0,0.95,41.873436,12.512979,"{'lat': 41.8734361, 'lon': 12.5129787, 'system..."


Conexión con la API de PVWatts para extraer los datos eléctricos

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime, timedelta

# --- CONFIGURACIÓN ---
API_KEY = "kbH0gYz3qPuqITp9labDcEGzVfujItUCMhrxWcol"
BASE_URL = "https://developer.nrel.gov/api/pvwatts/v8.json"
FEBRERO_DIAS = 28
DIAS_DEL_MES = list(range(1, FEBRERO_DIAS + 1))

# Inicializar un diccionario para almacenar los resultados diarios por estación
resultados_diarios = {}

# Suponemos que tu DataFrame se llama 'estaciones' y ya contiene la columna 'PVWatts_Params'
# con todos los parámetros necesarios (lat, lon, system_capacity, timeframe=hourly, etc.)

print(f"Iniciando {len(estaciones)} peticiones a la API de PVWatts...")
print("Nota: El proceso puede tardar varios minutos.")

# --------------------------------------------------------------------------------------
# 1. ITERAR SOBRE LAS ESTACIONES Y HACER LA PETICIÓN
# --------------------------------------------------------------------------------------

for index, row in estaciones.iterrows():
    estacion_id = index  # Usamos el índice de la fila como ID de la estación

    # Obtener los parámetros base y añadir la API Key
    parametros = row['PVWatts_Params'].copy()
    parametros['api_key'] = API_KEY

    print(f"-> Procesando Estación ID: {estacion_id} (Capacidad: {parametros['system_capacity']} kW)...")

    try:
        # Realizar la solicitud GET
        response = requests.get(BASE_URL, params=parametros)
        response.raise_for_status()  # Lanza excepción para errores 4xx/5xx

        datos = response.json()

        # Verificar si la respuesta contiene datos de salida
        if 'outputs' not in datos or 'ac' not in datos['outputs']:
            print(f"--- ERROR: La respuesta para la estación {estacion_id} no contiene datos horarios válidos.")
            continue

        # La producción horaria (Wh) son 8760 valores
        produccion_horaria_wh = datos['outputs']['ac']

        # --------------------------------------------------------------------------------------
        # 2. PROCESAR Y FILTRAR LOS DATOS PARA FEBRERO
        # --------------------------------------------------------------------------------------

        # PVWatts devuelve 8760 horas comenzando el 1 de enero a las 00:00 (índice 0)

        # Determinar el inicio y fin de febrero (día 32 a 59 para un año no bisiesto)
        # 31 días (Enero) * 24 horas/día = 744 horas

        # Inicio de Febrero (hora 0 de la noche del 1 de feb): Índice 744
        # Fin de Febrero (última hora del 28 de feb): Índice 744 + (28 * 24) - 1 = 1415

        INICIO_FEBRERO_INDEX = 744
        FIN_FEBRERO_INDEX = INICIO_FEBRERO_INDEX + (FEBRERO_DIAS * 24)

        produccion_febrero = produccion_horaria_wh[INICIO_FEBRERO_INDEX:FIN_FEBRERO_INDEX]


        # --------------------------------------------------------------------------------------
        # 3. CONVERTIR DATOS HORARIOS A FORMATO DIARIO (Diccionario por día)
        # --------------------------------------------------------------------------------------

        datos_febrero_diarios = {}

        for dia in DIAS_DEL_MES:
            # Calcular el rango de índices para este día
            inicio_dia_index = (dia - 1) * 24
            fin_dia_index = inicio_dia_index + 24

            # Extraer las 24 horas del día
            horas_del_dia = produccion_febrero[inicio_dia_index:fin_dia_index]

            # Crear el diccionario {hora: valor} para el día
            diccionario_diario = {hora: valor for hora, valor in enumerate(horas_del_dia)}

            # Almacenar el resultado para la estación
            datos_febrero_diarios[f"Día {dia}"] = diccionario_diario

        # Almacenar los resultados de la estación
        resultados_diarios[estacion_id] = datos_febrero_diarios

    except requests.exceptions.HTTPError as errh:
        print(f"--- ERROR HTTP para ID {estacion_id}: {errh}")
        resultados_diarios[estacion_id] = {f"Día {d}": f"ERROR HTTP: {errh.response.status_code}" for d in DIAS_DEL_MES}
    except requests.exceptions.RequestException as err:
        print(f"--- ERROR de Conexión para ID {estacion_id}: {err}")
        resultados_diarios[estacion_id] = {f"Día {d}": "ERROR CONEXIÓN" for d in DIAS_MES}
    except Exception as e:
        print(f"--- ERROR Inesperado para ID {estacion_id}: {e}")
        resultados_diarios[estacion_id] = {f"Día {d}": f"ERROR: {e}" for d in DIAS_MES}

print("\nProceso de peticiones y procesamiento finalizado.")

# --------------------------------------------------------------------------------------
# 4. CREAR EL DATAFRAME FINAL 'recarga_estaciones'
# --------------------------------------------------------------------------------------

# Convertir el diccionario de resultados en un DataFrame
recarga_estaciones = pd.DataFrame.from_dict(resultados_diarios, orient='index')

# Asegurar que el DataFrame tiene las 28 columnas de febrero
columnas_febrero = [f"Día {d}" for d in DIAS_DEL_MES]
recarga_estaciones = recarga_estaciones.reindex(columns=columnas_febrero)

# Establecer el nombre del índice para claridad
recarga_estaciones.index.name = 'Estacion_ID'

print("\n--- DataFrame 'recarga_estaciones' Creado ---")
print(recarga_estaciones.head())
print(f"\nDimensiones finales: {recarga_estaciones.shape}")

In [ ]:
# Asumimos que 'recarga_estaciones' ya está definido con los resultados de la API
# y que cada celda contiene el diccionario {hora: Wh}

# 1. Definición de la Potencia Requerida por un Puesto de Carga
POTENCIA_POR_PUESTO_W = 96 # W (48V * 2A = 96W)

# 2. Definición de la función de cálculo
def calcular_capacidad_horaria(diccionario_horario):
    """
    Toma un diccionario de energía horaria {hora: Wh} y lo convierte en
    un diccionario de capacidad de puestos de carga {hora: num_puestos}.
    """
    # Si la celda contiene un mensaje de error (string) en lugar de un diccionario, lo devuelve.
    if not isinstance(diccionario_horario, dict):
        return diccionario_horario

    capacidad_dict = {}
    for hora, energia_wh in diccionario_horario.items():
        # Capacidad = Energía producida (Wh) / Potencia consumida por puesto (W)
        # El resultado es el número de puestos que pueden recibir 1 hora de carga completa.
        capacidad = energia_wh / POTENCIA_POR_PUESTO_W
        capacidad_dict[hora] = capacidad

    return capacidad_dict

# 3. Crear el nuevo DataFrame aplicando la función a cada celda (Día)
# El método applymap aplica la función a CADA ELEMENTO (celda) del DataFrame.
capacidad_recarga = recarga_estaciones.applymap(calcular_capacidad_horaria)

# Establecer el nombre del índice
capacidad_recarga.index.name = 'Estacion_ID'

print("--- DataFrame 'capacidad_recarga' Creado ---")
print("Contiene el número equivalente de puestos de patinetes que pueden ser cargados por hora (Wh / 96W).")
capacidad_recarga

In [70]:
recarga_estaciones.to_csv("recarga_estaciones.csv")

In [71]:
capacidad_recarga.to_csv("capacidad_recarga.csv")